<a href="https://colab.research.google.com/github/amankiitg/LLM_Prod/blob/main/Prod_LLM_RAFT_Gary_Marcus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai chromadb tiktoken
import os, json, numpy as np
import openai
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

# Set your OpenAI API key (required for embedding & fine-tuning calls)
openai.api_key = ""


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.7 MB/s eta

In [2]:
import json

with open('garymarcus_transcript_1.json', 'r') as f:
    data = json.load(f)

# Pull out the exchanges list regardless of top-level shape
if isinstance(data, dict) and 'exchanges' in data:
    exchanges = data['exchanges']
elif isinstance(data, list):
    exchanges = data
else:
    raise ValueError("Unexpected JSON structure: expected a dict with 'exchanges' or a list.")

print(f"Total exchanges in transcript: {len(exchanges)}")

def to_qa(item):
    """Normalize one exchange to (question, answer)."""
    if isinstance(item, (list, tuple)) and len(item) >= 2:
        return item[0], item[1]
    if isinstance(item, dict):
        q = item.get('q') or item.get('question') or item.get('prompt') or item.get('Q')
        a = item.get('a') or item.get('answer')  or item.get('A')
        if q is None or a is None:
            raise KeyError("Exchange dict missing 'q'/'a' (or 'question'/'answer').")
        return q, a
    raise TypeError(f"Unrecognized exchange format: {type(item)}")

# Show the first Q/A
first_Q, first_A = to_qa(exchanges[0])
print("Sample Question:", first_Q[:100], "...")
print("Sample Answer:",   first_A[:100], "...")

# (Optional) Normalize everything and write JSONL for chat fine-tuning / RAG evals
qa_pairs = [to_qa(x) for x in exchanges]
with open('garymarcus_qa.jsonl', 'w', encoding='utf-8') as out:
    for q, a in qa_pairs:
        rec = {"messages": [{"role": "user", "content": q},
                            {"role": "assistant", "content": a}]}
        out.write(json.dumps(rec, ensure_ascii=False) + "\n")


Total exchanges in transcript: 27
Sample Question: So I want to begin in an experience that people are having, this sort of maybe first confrontation,  ...
Sample Answer: It’s synthesizing a bunch of stuff that humans have actually written already, sometimes for better a ...


In [3]:
import json, re

with open('garymarcus_finetune.json', 'r') as f:
    data = json.load(f)

def parse_memories(value):
    """Return a list of memory snippets from either a list or a 'from ...:' block string."""
    if not value:
        return []
    if isinstance(value, list):
        return [m.strip() for m in value if isinstance(m, str) and m.strip()]
    if isinstance(value, str):
        s = value.strip()
        # Capture text after each "from ...:" header until the next "from ...:" or end
        pat = re.compile(r'from[^\n]*?:\s*(.*?)(?=\n\s*from[^\n]*?:|\Z)', re.IGNORECASE | re.DOTALL)
        blocks = [m.group(1).strip().strip('"').strip() for m in pat.finditer(s)]
        return [b for b in blocks if b] if blocks else ([s] if s else [])
    return []

all_memories, seen = [], set()
example_count = 0

for item in data:
    ex = item.get("example")
    if not ex:
        continue  # skip the top-level metadata object
    example_count += 1
    sims = ex.get("similar_memories")
    for mem in parse_memories(sims):
        norm = re.sub(r"\s+", " ", mem).strip()
        if norm and norm not in seen:
            seen.add(norm)
            all_memories.append(norm)

print(f"Examples found: {example_count}")
print(f"Total unique memory snippets: {len(all_memories)}")
if all_memories:
    print("Example memory snippet:", all_memories[0][:80], "...")
else:
    print("No memory snippets found.")

# (Optional) Save a clean list, one per line
with open("garymarcus_memories.jsonl", "w", encoding="utf-8") as out:
    for m in all_memories:
        out.write(json.dumps({"memory": m}, ensure_ascii=False) + "\n")



Examples found: 71
Total unique memory snippets: 93
Example memory snippet: 03:26.829000+00:00: "ChatGPT is not understanding philosophy or economics — it's ...


In [ ]:
import chromadb
from chromadb.utils import embedding_functions
import openai

# 1. Initialize the Chroma client
chroma_client = chromadb.PersistentClient(path="./chroma_db")


# (Re)create the collection with an embedder attached
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    model_name="text-embedding-ada-002",  # you can switch to text-embedding-3-small
    api_key=openai.api_key,
)

# If you already created the collection earlier, recreate it:
try:
    chroma_client.delete_collection("gary_marcus_memories")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="gary_marcus_memories",
    embedding_function=openai_ef,
)

# Add all memories (Chroma will embed internally)
ids = [f"mem{idx}" for idx, _ in enumerate(all_memories)]
metadatas = [{"text": m} for m in all_memories]
collection.add(ids=ids, documents=all_memories, metadatas=metadatas)

print("Inserted", collection.count(), "memory embeddings into Chroma.")


In [ ]:
# --- Extract the first usable question from your 'data' structure ---
def first_question_from_list_or_dict(obj):
    """
    Returns the first non-empty question string from:
    - list of items where items may be dicts with an 'example' dict containing 'question'
    - dict with 'exchanges' (fallback)
    Returns None if no question is found.
    """
    # Case 1: dict with 'exchanges' (older format)
    if isinstance(obj, dict) and "exchanges" in obj:
        exs = obj.get("exchanges") or []
        for it in exs:
            if isinstance(it, (list, tuple)) and it:
                q = it[0]
            elif isinstance(it, dict):
                q = it.get("q") or it.get("question") or it.get("prompt") or it.get("Q")
            else:
                q = None
            if isinstance(q, str) and q.strip():
                return q.strip()
        return None

    # Case 2: list of mixed items (your current transcript)
    if isinstance(obj, list):
        for it in obj:
            if isinstance(it, dict):
                # Prefer the 'example' container if present
                ex = it.get("example")
                if isinstance(ex, dict):
                    q = ex.get("question") or ex.get("q") or ex.get("prompt") or ex.get("Q")
                    if isinstance(q, str) and q.strip():
                        return q.strip()
                # Fallback: maybe the dict itself has question fields
                q = it.get("question") or it.get("q") or it.get("prompt") or it.get("Q")
                if isinstance(q, str) and q.strip():
                    return q.strip()
        return None

    return None

sample_question = first_question_from_list_or_dict(data)
print("Sample question:", sample_question)


Sample question: So I want to begin in an experience that people are having, this sort of maybe first confrontation, for a lot of people, with these large language networks. When I ask ChatGPT, say, whether lower health care costs lead to higher employee wages or ask it to explain the Buddhist concept of emptiness, and it gives me pretty damn good answers, what is it actually doing?


In [ ]:
def retrieve_memories(question, k=2):
    if not isinstance(question, str) or not question.strip():
        raise ValueError(f"Invalid question: {question}")

    include = ["documents", "distances", "metadatas"]

    has_embedder = getattr(collection, "_embedding_function", None) is not None
    if has_embedder:
        res = collection.query(query_texts=[question], n_results=k, include=include)
    else:
        vec = openai_ef([question])[0]
        vec = vec.tolist() if hasattr(vec, "tolist") else vec
        res = collection.query(query_embeddings=[vec], n_results=k, include=include)

    ids  = res.get("ids", [[]])[0]
    docs = res.get("documents", [[]])[0]
    dsts = res.get("distances", [[]])[0]
    metas= res.get("metadatas", [[]])[0]

    out = []
    for i, doc in enumerate(docs):
        out.append({
            "id": ids[i] if i < len(ids) else None,
            "doc": doc,
            "distance": float(dsts[i]) if i < len(dsts) else None,
            "metadata": metas[i] if i < len(metas) else None,
        })
    return out


In [ ]:
if not sample_question:
    raise ValueError("No usable question found in transcript.")

print("Querying with:", sample_question)
topk = retrieve_memories(sample_question, k=2)
for r in topk:
    dist = r["distance"] if r["distance"] is not None else "NA"
    print(f"- id={r['id']} | dist={dist}\n  {r['doc']}\n")


Querying with: So I want to begin in an experience that people are having, this sort of maybe first confrontation, for a lot of people, with these large language networks. When I ask ChatGPT, say, whether lower health care costs lead to higher employee wages or ask it to explain the Buddhist concept of emptiness, and it gives me pretty damn good answers, what is it actually doing?
- id=mem0 | dist=0.2873981297016144
  03:26.829000+00:00: "ChatGPT is not understanding philosophy or economics — it's guessing what it should say based on patterns it has seen before in the data it was trained on.

- id=mem56 | dist=0.32173511385917664
  ChatGPT, and AI systems like it, work based on existing data or text they've been trained on. They do not genuinely comprehend or connect to the world in the way humans do. Their ability to 'write' or 'speak' is based on finding patterns in the training data, much like learning to predict the next word in a sentence based on the words that came before. As fo

In [ ]:
import json, re
finetune_data = data  # alias so your later loop works


# Parse "similar_memories" whether it's a list or a "from ...:" string
def parse_memories(value):
    if not value:
        return []
    if isinstance(value, list):
        return [m.strip() for m in value if isinstance(m, str) and m.strip()]
    if isinstance(value, str):
        s = value.strip()
        pat = re.compile(r'from[^\n]*?:\s*(.*?)(?=\n\s*from[^\n]*?:|\Z)', re.IGNORECASE | re.DOTALL)
        blocks = [m.group(1).strip().strip('"').strip() for m in pat.finditer(s)]
        return [b for b in blocks if b] or ([s] if s else [])
    return []

training_file = "gary_marcus_ft_train.jsonl"
written = 0

with open(training_file, 'w', encoding='utf-8') as f:
    for item in finetune_data:
        ex = item.get("example")
        if not ex:
            continue  # skip metadata rows

        q = ex.get("question") or ex.get("q")
        a = ex.get("answer")  or ex.get("a")
        if not q or not a:
            continue

        mem_list = parse_memories(ex.get("similar_memories"))
        context_text = "\n".join(mem_list) if mem_list else ""

        user_prompt = (f"Relevant information:\n{context_text}\n\n" if context_text else "") + f"Question: {q}"

        messages = [
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": a}
        ]
        f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        written += 1

print(f"Wrote {written} training lines to {training_file}")

# Peek at the first example
with open(training_file, 'r', encoding='utf-8') as f:
    print("First training example:", f.readline()[:300], "...")


Wrote 71 training lines to gary_marcus_ft_train.jsonl
First training example: {"messages": [{"role": "user", "content": "Relevant information:\n03:26.829000+00:00: \n \"ChatGPT is not understanding philosophy or economics — it's guessing what it should say based on patterns it has seen before in the data it was trained on.\n\nQuestion: So I want to begin in an experience that ...


In [ ]:
# ✅ Set API key for this kernel session (won't print it)
import os
from getpass import getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI
client = OpenAI()  # now picks up OPENAI_API_KEY


Enter your OpenAI API key: ··········


In [ ]:
# pip install --upgrade openai
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY from env

training_file = "gary_marcus_ft_train.jsonl"

# 1) Upload your JSONL
file_obj = client.files.create(
    file=open(training_file, "rb"),
    purpose="fine-tune",
)
print("Uploaded file id:", file_obj.id)

# 2) Create a fine-tuning job (pick a supported model)
job = client.fine_tuning.jobs.create(
    training_file=file_obj.id,
    model="gpt-3.5-turbo",  # or "gpt-4.1-mini" / "gpt-4.1"
    # optional:
    hyperparameters={"n_epochs": 1},
    # suffix="gary-marcus"
)
print("Job id:", job.id, "| status:", job.status)

# 3) (Optional) Inspect latest events
events = client.fine_tuning.jobs.list_events(job.id, limit=10)
for e in events.data[::-1]:
    print(f"[{e.created_at}] {e.message}")

# 4) (Later) get the resulting model id, then use it
job = client.fine_tuning.jobs.retrieve(job.id)
ft_model = job.fine_tuned_model
print("Fine-tuned model:", ft_model)

# Example inference (Chat Completions; Responses API also works)
if ft_model:
    resp = client.chat.completions.create(
        model=ft_model,
        messages=[{"role": "user", "content": "Give me a one-sentence summary of your domain."}]
    )
    print(resp.choices[0].message.content)



Uploaded file id: file-588d8aF3nuPyDxyXnru4CU
Job id: ftjob-dpRj9OK0UWPqq18NBJLDD2Xq | status: validating_files
[1758085205] Created fine-tuning job: ftjob-dpRj9OK0UWPqq18NBJLDD2Xq
[1758085205] Validating training file: file-588d8aF3nuPyDxyXnru4CU
Fine-tuned model: None


In [ ]:
import re, difflib


#FT_MODEL = "ft:gpt-3.5-turbo-0125:vizuara::CFzhoKmJ"
FT_MODEL = "ft:gpt-3.5-turbo-0125:vizuara::CGee0CJW"

BANNED_PHRASES = [
    "drives the cost of plausible misinformation toward zero",
    "flooding channels with confident text that lacks grounding in truth",
]

def _normalize(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def _too_similar(ans: str, query: str, context: str) -> bool:
    a = _normalize(ans)
    if not a:
        return True
    # 1) direct banned phrases
    if any(p in a for p in BANNED_PHRASES):
        return True
    # 2) echoes the question
    q = _normalize(query)
    if q and q in a:
        return True
    # 3) long overlap with context (rough heuristic)
    m = difflib.SequenceMatcher(None, a, _normalize(context))
    longest = max((blk.size for blk in m.get_matching_blocks()), default=0)
    return longest >= 80  # ~80+ chars contiguous overlap → likely copy

def answer_question(query, model_name=FT_MODEL, k=2, temperature=0.2, max_tokens=2000):
    # 1) Retrieve top-k memories
    hits = retrieve_memories(query, k=k)
    def hit_to_text(h):
        if isinstance(h, dict):
            return h.get("doc") or h.get("text") or h.get("document") or ""
        return str(h or "")
    mem_texts = [t.strip() for t in map(hit_to_text, hits) if t and t.strip()]
    context = "\n\n".join(mem_texts) if mem_texts else "(No relevant memory found)"

    # 2) Compose messages (no few-shot)
    system_msg = {
        "role": "system",
        "content": (
            "You are Gary Marcus. Answer clearly, skeptically, and precisely. "
            "Do NOT repeat or paraphrase the user's question. "
            "Do NOT copy sentences from CONTEXT; paraphrase ideas only. "
            "Prefer concrete risks and mechanisms; include one short mitigation."
        ),
    }
    user_msg = {
        "role": "user",
        "content": (
            f"CONTEXT (paraphrase only, do not quote):\n{context}\n\n"
            f"QUESTION: {query}\n\n"
            "Write 3–6 sentences. Start directly with the answer."
        ),
    }

    # 3) Generate 2 candidates and pick the less echo-y one
    resp = client.chat.completions.create(
        model=model_name,
        messages=[system_msg, user_msg],
        temperature=temperature,
        max_tokens=max_tokens,
        n=2,
        frequency_penalty=0.2,
        presence_penalty=0.0,
        stop=["QUESTION:", "Context:", "CONTEXT:"],
    )
    cands = [c.message.content.strip() for c in resp.choices]
    scored = [(i, _too_similar(a, query, context)) for i, a in enumerate(cands)]
    # prefer any non-echo candidate
    non_echo = [cands[i] for i, bad in scored if not bad]
    answer = non_echo[0] if non_echo else cands[0]

    # 4) If still echoing, retry once with stronger constraint
    if _too_similar(answer, query, context):
        user_msg["content"] += (
            "\n\nRewrite without reusing any distinctive phrases from earlier. "
            "Avoid these exact words: " + "; ".join(BANNED_PHRASES) + "."
        )
        resp2 = client.chat.completions.create(
            model=model_name,
            messages=[system_msg, user_msg],
            temperature=0.3,
            max_tokens=max_tokens,
            n=1,
            frequency_penalty=0.3,
            presence_penalty=0.0,
            stop=["QUESTION:", "Context:", "CONTEXT:"],
        )
        answer = resp2.choices[0].message.content.strip()

    return context, answer

# Example
test_question = "What dangers do large language models pose in terms of misinformation?"
mem, ans = answer_question(test_question, model_name=FT_MODEL, k=2)

from IPython.display import Markdown, display
display(Markdown(f"**Question:** {test_question}\n\n**Retrieved Memory:**\n> {mem}\n\n**Answer:** {ans}"))


**Question:** What dangers do large language models pose in terms of misinformation?

**Retrieved Memory:**
> 45:07.577000+00:00: Large language models are akin to jack-of-all-trades yet masters of none, creating an appearance of great capacity, often without delivering. While these models may provide answers on a wide array of topics, it doesn't always mean they have a deep understanding of the subject matter. Indeed, they can be viewed as unreliable tools, providing an illusion of great versatility without necessarily delivering on these promises. This change in technology, where such models have become pervasive, has significantly altered our interactions and society itself.

believe that these AI systems pose a major threat due to their lack of understanding truth. If AI can proliferate misinformation cheaply and profusely, the line between truth and falsehood may become indistinguishable. It's analogous to a 'Jurassic Park moment' where the price of spreading misleading information is drastically reduced, leaving society exposed to mass manipulation and confusion.

**Answer:** They can proliferate misinformation cheaply and profusely, making it difficult to distinguish truth from falsehood.

In [ ]:
!pip install -q --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 29.4 MB/s eta 0:00:00


In [ ]:
# pip install gradio --quiet
import re, difflib, textwrap
import gradio as gr

# ==== CONFIG ====
#FT_MODEL = "ft:gpt-3.5-turbo-0125:vizuara::CFzhoKmJ"  # your fine-tuned model id
FT_MODEL = "ft:gpt-3.5-turbo-0125:vizuara::CGee0CJW"

# If `client` (OpenAI v1) isn't defined yet, uncomment:
# from getpass import getpass
# import os
# from openai import OpenAI
# if not os.getenv("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
# client = OpenAI()

# ----- Helpers (anti-echo + formatting) -----
BANNED_PHRASES = []  # add any phrases you never want echoed here

def _normalize(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def _too_similar(ans: str, query: str, context: str) -> bool:
    a = _normalize(ans)
    if not a:
        return True
    if any(p in a for p in BANNED_PHRASES):
        return True
    if _normalize(query) and _normalize(query) in a:
        return True
    # Large contiguous overlap with context -> likely copy
    m = difflib.SequenceMatcher(None, a, _normalize(context))
    longest = max((blk.size for blk in m.get_matching_blocks()), default=0)
    return longest >= 120

def _hit_to_text(h):
    # Works with your retrieve_memories() which returns dicts,
    # and also with plain strings (older versions)
    if isinstance(h, dict):
        return h.get("doc") or h.get("text") or h.get("document") or ""
    return str(h or "")

def _format_citations(hits):
    lines = []
    for h in hits:
        if isinstance(h, dict):
            hid = h.get("id", "—")
            dist = h.get("distance", None)
            txt = _hit_to_text(h)
        else:
            hid, dist, txt = "—", None, _hit_to_text(h)
        snippet = textwrap.shorten(txt.replace("\n", " "), width=200, placeholder="…")
        if dist is None:
            lines.append(f"- **[{hid}]** {snippet}")
        else:
            lines.append(f"- **[{hid}]** (distance: `{dist:.4f}`) — {snippet}")
    return "\n".join(lines) if lines else "_No citations_"

# ----- Core RAG function -----
def rag_answer(query, k=2, temperature=0.2, max_tokens=2000, model_name=FT_MODEL):
    # 1) Retrieve memories
    hits = retrieve_memories(query, k=int(k))  # uses your existing function
    mem_texts = [_hit_to_text(h).strip() for h in hits if _hit_to_text(h).strip()]
    context = "\n\n".join(mem_texts) if mem_texts else "(No relevant memory found)"
    citations_md = _format_citations(hits)

    # 2) Compose messages (no few-shot; anti-echo rules in system)
    system_msg = {
        "role": "system",
        "content": (
            "You are Gary Marcus. Answer clearly, skeptically, and precisely. "
            "Do NOT repeat or paraphrase the user's question. "
            "Do NOT copy sentences from CONTEXT; paraphrase ideas only. "
            "Prefer concrete risks and mechanisms; include one brief mitigation."
        ),
    }
    user_msg = {
        "role": "user",
        "content": (
            f"CONTEXT (paraphrase only, do not quote):\n{context}\n\n"
            f"QUESTION: {query}\n\n"
            "Write 3–6 sentences. Start directly with the answer."
        ),
    }

    # 3) Generate 2 candidates; pick the one that isn't echoing
    resp = client.chat.completions.create(
        model=model_name,
        messages=[system_msg, user_msg],
        temperature=float(temperature),
        max_tokens=int(max_tokens),
        n=2,
        frequency_penalty=0.2,
        presence_penalty=0.0,
        stop=["QUESTION:", "Context:", "CONTEXT:"],
    )
    cands = [c.message.content.strip() for c in resp.choices]
    non_echo = [a for a in cands if not _too_similar(a, query, context)]
    answer = (non_echo[0] if non_echo else cands[0]).strip()

    # 4) If still echoing, retry once with stronger instruction
    if _too_similar(answer, query, context):
        user_msg["content"] += (
            "\n\nRewrite without reusing distinctive phrases from earlier. "
            "Avoid verbatim reuse of any sequence longer than 12 words from CONTEXT."
        )
        resp2 = client.chat.completions.create(
            model=model_name,
            messages=[system_msg, user_msg],
            temperature=min(0.3, float(temperature) + 0.1),
            max_tokens=int(max_tokens),
            n=1,
            frequency_penalty=0.3,
            presence_penalty=0.0,
            stop=["QUESTION:", "Context:", "CONTEXT:"],
        )
        answer = resp2.choices[0].message.content.strip()

    return answer, citations_md, context  # for optional display

# ----- Gradio UI -----
with gr.Blocks(title="RAFT:Retrieval-Augmented Fine-Tuning") as demo:
    gr.Markdown(
        """
        <div style="text-align: center">
            <h1>RAFT: Retrieval Augmented Fine-Tuning</h1>
            <p style="font-size:1.05rem; margin-top:-0.6rem;">
                A new way to teach LLMs to be better at RAG
            </p>
        </div>
        """
    )
    gr.Markdown(
        "Type a question. The app retrieves top-k memory snippets from Chroma and queries your fine-tuned model. "
        "You’ll see the answer and the citations (IDs, distances, and snippets)."
    )

    with gr.Row():
        q_in = gr.Textbox(label="Your question", placeholder="Ask anything related to the interview/topics…", lines=3)
    with gr.Row():
        k_in = gr.Slider(1, 5, value=2, step=1, label="Top-k memories")
        temp_in = gr.Slider(0.0, 1.0, value=0.2, step=0.05, label="Temperature")
        mtok_in = gr.Slider(150, 1000, value=450, step=50, label="Max tokens")
    with gr.Accordion("Advanced", open=False):
        model_in = gr.Textbox(value=FT_MODEL, label="Model (fine-tuned)", interactive=True)

    ask_btn = gr.Button("Ask", variant="primary")
    with gr.Row():
        ans_out = gr.Markdown(label="Answer")
    with gr.Row():
        cites_out = gr.Markdown(label="Citations (smaller distance = closer)")
    with gr.Accordion("Context used (raw, concatenated)", open=False):
        ctx_out = gr.Textbox(label="Context", lines=8)

    def _on_click(q, k, t, mtok, model):
        if not q or not q.strip():
            return "Please enter a question.", "_No citations_", ""
        try:
            ans, cites, ctx = rag_answer(q.strip(), k=k, temperature=t, max_tokens=mtok, model_name=model.strip())
            return ans, cites, ctx
        except Exception as e:
            return f"⚠️ Error: {e}", "_No citations_", ""

    ask_btn.click(_on_click, inputs=[q_in, k_in, temp_in, mtok_in, model_in], outputs=[ans_out, cites_out, ctx_out])

    gr.Examples(
        examples=[
            ["What dangers do large language models pose in terms of misinformation?"],
            ["Why do you say pastiche isn’t understanding?"],
            ["How could we mitigate AI-driven misinformation?"],
        ],
        inputs=[q_in],
        label="Examples",
    )

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f97b9c6e57fd5090f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f97b9c6e57fd5090f1.gradio.live
